In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
target_df=spark.table('voltgrid.silver.sessions')

In [0]:
df=spark.table('voltgrid.bronze.sessions')

In [0]:
source_df=df.withColumn('rnk',F.row_number().over(Window.partitionBy('session_id').orderBy(F.desc('ingestion_time')))).filter(F.col('rnk')==1).withColumn('connector_type',F.upper(F.trim('connector_type'))).withColumn('session_status',F.upper(F.trim('session_status'))).drop('ingestion_time','rnk','source_path')

####  Start `SCD Type 2`Process

In [0]:
history_records=target_df.filter(F.col('active_flg')=='N')

In [0]:
hash_keys=['vehicle_id','station_id','customer_id','started_at','ended_at','duration_min','energy_kwh','cost_aud','peak_power_kw','connector_type','session_status']

source_df=source_df.withColumn('hash_key',F.xxhash64(*[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in hash_keys]))
target_df=target_df.withColumn('hash_key',F.xxhash64(*[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in hash_keys]))



In [0]:
change_records=target_df.alias('tr').join(source_df.alias('sr'),F.col("tr.session_id")==F.col("sr.session_id")).filter(F.col('tr.hash_key')!=F.col('sr.hash_key')).select('tr.*').withColumn('active_flg',F.lit('N')).withColumn('end_date',F.current_date()).drop('hash_key')

In [0]:
source_df=source_df.drop("hash_key")
target_df=target_df.drop("hash_key")

In [0]:
old_records=target_df.alias('tr').join(change_records.alias('sr'),F.col("tr.session_id")==F.col("sr.session_id"),'left_anti')

In [0]:
change_records_val=source_df.alias('sr').join(change_records.alias('tr'),F.col("sr.session_id")==F.col("tr.session_id"),'left_semi').withColumn('active_flg',F.lit('Y')).withColumn('start_date',F.current_date()).withColumn('end_date',F.lit(None).cast('date'))

In [0]:
new_records=source_df.alias('sr').join(change_records.alias('tr'),F.col("sr.session_id")==F.col("tr.session_id"),'left_anti').withColumn('active_flg',F.lit('Y')).withColumn('start_date',F.current_date()).withColumn('end_date',F.lit(None).cast('date'))

In [0]:
total_records = (
    history_records
    .unionByName(change_records)
    .unionByName(change_records_val)
    .unionByName(old_records)
    .unionByName(new_records)
)

In [0]:
total_records.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable('voltgrid.silver.sessions')